# Notebook 02 — Feature Engineering
**Mental Health Assessment Using Machine Learning**

---

## Purpose

This notebook applies nine feature selection methods to identify the 15 most informative features from the 26 validated psychometric scale items. The output is nine distinct feature sets — each containing a scaled training array, a scaled test array, and a fitted scaler.

---

## Feature Scope

All 9 methods operate on the **26 psychometric scale items only**. Demographic columns (`Age`, `Gender`, `University`, `Department`, `Year`, `CGPA`, `Scholarship`), composite totals (`PSS Total`, `GAD Total`, `PHQ Total`), and per-scale class labels (`Stress Level`, `Anxiety Level`, `Depression Level`) are excluded.

- Totals are linear combinations of the items and would introduce multicollinearity
- Per-scale class labels are derived from the target and would cause data leakage
- Demographics add noise that obscures the psychometric signal

| Scale | Items | Response range |
|-------|-------|---------------|
| PSS-10 | PSS1–PSS10 | 0–4 (PSS5,6,7,8 already reverse-scored) |
| GAD-7  | GAD1–GAD7  | 0–3 |
| PHQ-9  | PHQ1–PHQ9  | 0–3 |

---

## Feature Selection Methods

Each method selects exactly **15 features** from the 26.

| Key | Method | Criterion |
|-----|--------|----------|
| `rfe` | Recursive Feature Elimination | Iteratively removes the weakest feature by a Random Forest estimator's importance score until 15 remain |
| `skb` | Select K Best — ANOVA F-statistic | Ranks features by the one-way ANOVA F-score between feature values and class labels; selects top 15 |
| `fscs` | Fisher Score — Chi-squared Test | Ranks by chi-squared statistic (observed vs expected co-occurrence with class); requires non-negative input — satisfied by all 26 items |
| `etc` | Extra Trees Classifier | Ranks by Gini-based mean impurity decrease from a fitted ExtraTreesClassifier; selects top 15 |
| `pc` | Pearson Correlation | Absolute Pearson correlation coefficient between each feature and the ordinal target (Stable=0, Challenged=1, Critical=2); selects top 15 |
| `mi` | Mutual Information — Classification | Estimates mutual information between each feature and the class target using `mutual_info_classif`; selects top 15 |
| `mir` | Mutual Information Regression | Same as MI but uses `mutual_info_regression`, treating the ordinal target as continuous; produces a different ranking |
| `mu` | Manual Uniqueness | Counts unique values per feature on the pre-SMOTE training set; selects the 15 features with the fewest unique values — highly discrete features carry more concentrated information |
| `vt` | Variance Threshold | Selects top 15 features by variance on the SMOTE-resampled training set — higher variance indicates greater distributional spread |

---

## Processing Pipeline (per method)

```
X_train_sm (26 features) → feature selector → 15 indices
     ↓
Subset X_train_sm[:, indices] and X_test[:, indices]
     ↓
Fit StandardScaler on training subset → transform train and test
     ↓
Save train.csv (scaled + label), test.csv (scaled + label), scaler.pkl
```

**`mu` exception:** Unique value counts are computed on `X_train` (pre-SMOTE), not `X_train_sm`. SMOTE interpolation between integer values creates decimal counts that inflate unique value totals and would misrepresent the true data discreteness.

---

## Cell Map

| Cell | Summary |
|------|---------|
| 1 | Imports |
| 2 | Load dataset · define feature scope · encode target · stratified split · SMOTE · non-negativity assertion |
| 3 | Apply all 9 feature selection methods · scale · save feature sets |

## Cell 1 — Imports and Working Directory

Imports all libraries required for feature selection and preprocessing.

In [1]:
from pathlib import Path
import os, warnings
warnings.filterwarnings('ignore')

_cwd = Path.cwd()
if _cwd.name == 'notebooks':
    os.chdir(_cwd.parent)
print(f'Working directory: {Path.cwd()}')

import numpy as np
import pandas as pd
import joblib

from sklearn.preprocessing     import StandardScaler
from sklearn.model_selection   import train_test_split
from sklearn.feature_selection import (RFE, SelectKBest, f_classif, chi2,
                                        mutual_info_classif, mutual_info_regression)
from sklearn.ensemble          import RandomForestClassifier, ExtraTreesClassifier
from imblearn.over_sampling    import SMOTE

METHODS    = ['rfe', 'skb', 'fscs', 'etc', 'pc', 'mi', 'mir', 'mu', 'vt']
N_FEATURES = 15
SEED       = 42

print('\n✓ All imports successful')

Working directory: d:\Programming\Projects\Mental Health Assessment

✓ All imports successful


## Cell 2 — Load Dataset · Define Feature Scope · Split · SMOTE

Loads `mha_tabular_dataset.csv` (produced by Notebook 01 with the 40-column schema) and extracts the 26-item feature matrix. The target column `Mental Health Status` is encoded to integers: `Stable=0`, `Challenged=1`, `Critical=2`.

The data is split into a stratified 80% training set and a 20% held-out test set (`random_state=42`). Stratification ensures all three classes appear in both sets proportionally. SMOTE is then applied **only to the training set** to oversample the minority classes — applying SMOTE before the split would leak synthetic samples into the test set and produce optimistically biased evaluation metrics.

Two training arrays are retained:
- `X_train_sm` — SMOTE-resampled training set used by all selectors except `mu`
- `X_train` — the original pre-SMOTE training set used by `mu` to count real unique values

A non-negativity assertion confirms that all values in `X_train_sm` and `X_test` are ≥ 0, which is required for the chi-squared test (`fscs`). This is guaranteed by the PSS items (0–4) and GAD/PHQ items (0–3), and is preserved by SMOTE interpolation.

In [2]:
df = pd.read_csv(os.path.join('data', 'processed', 'mha_tabular_dataset.csv'))
print(f'Loaded : {df.shape}')

# 26 psychometric scale items only
FEATURE_COLS = (
    [f'PSS{i}' for i in range(1, 11)] +   # PSS1-PSS10  (0-4)
    [f'GAD{i}' for i in range(1,  8)] +   # GAD1-GAD7   (0-3)
    [f'PHQ{i}' for i in range(1, 10)]     # PHQ1-PHQ9   (0-3)
)  # 26 features total

# Encode target: Stable=0, Challenged=1, Critical=2
TARGET_MAP = {'Stable': 0, 'Challenged': 1, 'Critical': 2}
X = df[FEATURE_COLS].values.astype(float)
y = df['Mental Health Status'].map(TARGET_MAP).values

# Stratified 80/20 split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

# SMOTE on training set only — X_train retained for mu method
smote = SMOTE(random_state=SEED)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

# Non-negativity check required by chi2 (fscs)
assert X_train_sm.min() >= 0, 'Negative values in X_train_sm — chi2 will fail'
assert X_test.min()     >= 0, 'Negative values in X_test — chi2 will fail'

print(f'Features           : {len(FEATURE_COLS)} — {FEATURE_COLS}')
print(f'X_train  (raw)     : {X_train.shape}')
print(f'X_train  (SMOTE)   : {X_train_sm.shape}')
print(f'X_test             : {X_test.shape}')
print(f'y_train class dist : Stable={np.sum(y_train_sm==0)}, '
      f'Challenged={np.sum(y_train_sm==1)}, Critical={np.sum(y_train_sm==2)}')
print(f'y_test  class dist : Stable={np.sum(y_test==0)}, '
      f'Challenged={np.sum(y_test==1)}, Critical={np.sum(y_test==2)}')
print('\n✓ Non-negativity check passed — chi2 safe to use')

Loaded : (2022, 40)
Features           : 26 — ['PSS1', 'PSS2', 'PSS3', 'PSS4', 'PSS5', 'PSS6', 'PSS7', 'PSS8', 'PSS9', 'PSS10', 'GAD1', 'GAD2', 'GAD3', 'GAD4', 'GAD5', 'GAD6', 'GAD7', 'PHQ1', 'PHQ2', 'PHQ3', 'PHQ4', 'PHQ5', 'PHQ6', 'PHQ7', 'PHQ8', 'PHQ9']
X_train  (raw)     : (1617, 26)
X_train  (SMOTE)   : (3099, 26)
X_test             : (405, 26)
y_train class dist : Stable=1033, Challenged=1033, Critical=1033
y_test  class dist : Stable=25, Challenged=121, Critical=259

✓ Non-negativity check passed — chi2 safe to use


## Cell 3 — Apply 9 Feature Selection Methods and Save Feature Sets

Iterates over all 9 methods. For each:

1. **Select indices** — computes relevance scores using the method-specific criterion and identifies the 15 best feature indices from the 26 available. Indices are sorted by column position to ensure consistent column ordering across train and test CSVs.
2. **Subset** — extracts the 15 selected columns from both `X_train_sm` and `X_test`.
3. **Scale** — fits a `StandardScaler` on the selected training subset and applies it to both train and test. The scaler is fitted on training data only, never on the test set, to prevent data leakage during evaluation.
4. **Save** — writes three files to `features/Tabular/{method}/`: `train.csv` (scaled features + integer label column), `test.csv`, and `scaler.pkl`. The label column name in both CSVs is `label` (0/1/2).
5. **Store** — saves the scaled numpy arrays and selected column names in the `feature_sets` dict for inspection and logging.

In [3]:
feature_sets = {}

for method in METHODS:
    print(f'\n── {method.upper()} ──')

    # ── Select 15 feature indices ─────────────────────────────────────────────
    if method == 'rfe':
        sel = RFE(
            estimator=RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1),
            n_features_to_select=N_FEATURES
        )
        sel.fit(X_train_sm, y_train_sm)
        indices = sorted(np.where(sel.support_)[0].tolist())

    elif method == 'skb':
        sel = SelectKBest(f_classif, k=N_FEATURES)
        sel.fit(X_train_sm, y_train_sm)
        indices = sorted(sel.get_support(indices=True).tolist())

    elif method == 'fscs':
        # Fisher Score Chi-squared Test — non-negative input verified in Cell 2
        sel = SelectKBest(chi2, k=N_FEATURES)
        sel.fit(X_train_sm, y_train_sm)
        indices = sorted(sel.get_support(indices=True).tolist())

    elif method == 'etc':
        clf = ExtraTreesClassifier(n_estimators=100, random_state=SEED, n_jobs=-1)
        clf.fit(X_train_sm, y_train_sm)
        indices = sorted(np.argsort(clf.feature_importances_)[::-1][:N_FEATURES].tolist())

    elif method == 'pc':
        # Absolute Pearson correlation with ordinal numeric target
        target_f = y_train_sm.astype(float)
        corrs = np.array([
            abs(np.corrcoef(X_train_sm[:, i], target_f)[0, 1])
            for i in range(X_train_sm.shape[1])
        ])
        indices = sorted(np.argsort(np.nan_to_num(corrs))[::-1][:N_FEATURES].tolist())

    elif method == 'mi':
        scores  = mutual_info_classif(X_train_sm, y_train_sm, random_state=SEED)
        indices = sorted(np.argsort(scores)[::-1][:N_FEATURES].tolist())

    elif method == 'mir':
        # Target treated as continuous ordinal variable
        scores  = mutual_info_regression(
            X_train_sm, y_train_sm.astype(float), random_state=SEED
        )
        indices = sorted(np.argsort(scores)[::-1][:N_FEATURES].tolist())

    elif method == 'mu':
        # Count unique values on pre-SMOTE X_train (real data only)
        # Stable sort: ties broken by column index (ascending order)
        unique_counts = np.array([
            len(np.unique(X_train[:, i])) for i in range(X_train.shape[1])
        ])
        indices = sorted(np.argsort(unique_counts, kind='stable')[:N_FEATURES].tolist())

    elif method == 'vt':
        # Top 15 by variance on SMOTE-resampled training set
        variances = X_train_sm.var(axis=0)
        indices   = sorted(np.argsort(variances)[::-1][:N_FEATURES].tolist())

    # ── Subset, scale, and save ───────────────────────────────────────────────
    selected_cols = [FEATURE_COLS[i] for i in indices]
    X_tr_sel      = X_train_sm[:, indices]
    X_te_sel      = X_test[:, indices]

    scaler  = StandardScaler()
    X_tr_sc = scaler.fit_transform(X_tr_sel)
    X_te_sc = scaler.transform(X_te_sel)

    feat_dir = os.path.join('features', 'Tabular', method)
    os.makedirs(feat_dir, exist_ok=True)

    train_df = pd.DataFrame(X_tr_sc, columns=selected_cols)
    train_df['label'] = y_train_sm
    train_df.to_csv(os.path.join(feat_dir, 'train.csv'), index=False)

    test_df = pd.DataFrame(X_te_sc, columns=selected_cols)
    test_df['label'] = y_test
    test_df.to_csv(os.path.join(feat_dir, 'test.csv'), index=False)

    joblib.dump(scaler, os.path.join(feat_dir, 'scaler.pkl'))

    feature_sets[method] = {
        'train_X': X_tr_sc, 'train_y': y_train_sm,
        'test_X' : X_te_sc, 'test_y' : y_test,
        'cols'   : selected_cols
    }

    print(f'  Selected ({N_FEATURES}): {selected_cols}')
    print(f'  Train : {X_tr_sc.shape}  |  Test : {X_te_sc.shape}')
    print(f'  Saved → {feat_dir}/')

print('\n── Summary ──')
print(f'✓ {len(feature_sets)} feature sets saved to features/Tabular/')
for m, fs in feature_sets.items():
    print(f'  {m.upper():<5}: {fs["cols"]}')


── RFE ──
  Selected (15): ['PSS1', 'PSS2', 'GAD1', 'GAD2', 'GAD3', 'GAD4', 'GAD5', 'GAD6', 'GAD7', 'PHQ2', 'PHQ3', 'PHQ4', 'PHQ5', 'PHQ6', 'PHQ7']
  Train : (3099, 15)  |  Test : (405, 15)
  Saved → features\Tabular\rfe/

── SKB ──
  Selected (15): ['PSS2', 'PSS3', 'GAD1', 'GAD3', 'GAD4', 'GAD5', 'GAD6', 'GAD7', 'PHQ2', 'PHQ3', 'PHQ4', 'PHQ5', 'PHQ6', 'PHQ7', 'PHQ8']
  Train : (3099, 15)  |  Test : (405, 15)
  Saved → features\Tabular\skb/

── FSCS ──
  Selected (15): ['GAD1', 'GAD2', 'GAD3', 'GAD4', 'GAD5', 'GAD6', 'GAD7', 'PHQ2', 'PHQ3', 'PHQ4', 'PHQ5', 'PHQ6', 'PHQ7', 'PHQ8', 'PHQ9']
  Train : (3099, 15)  |  Test : (405, 15)
  Saved → features\Tabular\fscs/

── ETC ──
  Selected (15): ['PSS2', 'GAD1', 'GAD2', 'GAD3', 'GAD4', 'GAD5', 'GAD6', 'GAD7', 'PHQ2', 'PHQ3', 'PHQ4', 'PHQ5', 'PHQ6', 'PHQ7', 'PHQ8']
  Train : (3099, 15)  |  Test : (405, 15)
  Saved → features\Tabular\etc/

── PC ──
  Selected (15): ['PSS2', 'PSS3', 'GAD1', 'GAD3', 'GAD4', 'GAD5', 'GAD6', 'GAD7', 'PHQ2', 'PHQ3'